In [1]:
import torch, torch.onnx, tqdm, time, os, json
import numpy as np
import pandas as pd
import pytorch_lightning as pl
from models.est_model import ResidualRegression, DNNRegression
from torch.utils.data import DataLoader, Dataset, random_split
from pytorch_lightning.callbacks import DeviceStatsMonitor, EarlyStopping
from sklearn.metrics import r2_score

In [2]:
monitor = DeviceStatsMonitor()

In [3]:
# set train setting
save_model = True
train_num_epoch = 50000
min_loss = 100
dl_workers = 0

In [4]:
gpu = torch.device('cuda')
torch.set_float32_matmul_precision('high')

In [5]:
# load hyperparameter of json file
with open('.' + os.sep + os.path.join('models', 'params_dnn_20220207-012403.json'), 'r') as file:
    hyper_params = json.load(file)

n_inputs = hyper_params['n_of_inputs']
n_outputs = hyper_params['n_of_outputs']
n_layers = hyper_params['n_of_layers']

In [6]:
model = DNNRegression(n_inputs, n_layers, n_outputs)
print(model)

DNNRegression(
  (layer_1): Linear(in_features=4, out_features=20, bias=True)
  (layer_2): Linear(in_features=20, out_features=20, bias=True)
  (layer_3): Linear(in_features=20, out_features=6, bias=True)
  (active): ReLU()
  (model): Sequential(
    (0): Linear(in_features=4, out_features=20, bias=True)
    (1): ReLU()
    (2): Linear(in_features=20, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=6, bias=True)
  )
  (loss): MSELoss()
)


In [7]:
data = pd.read_csv('./resources/sim_data.csv')
feature_names = ['lift_weight(ton)', 'lift_height(m)', 'rising_angle(deg)', 'swing_angle(deg)']
target_names = ['left_ground_pressure_min(kg/cm2)', 'left_ground_pressure_max(kg/cm2)', 'left_pressure_length(m)', 'right_ground_pressure_min(kg/cm2)', 'right_ground_pressure_max(kg/cm2)', 'right_pressure_length(m)']

feature_data = []
target_data = []

for feature_name in feature_names:
    feature_data.append(data[feature_name])

for target_name in target_names:
    target_data.append(data[target_name])

feature_data = np.array(feature_data, dtype=np.float32).T
target_data = np.array(target_data, dtype=np.float32).T

class TensorDataset(Dataset):
    def __init__(self, feature, target):
        self.x_data = torch.tensor(feature)
        self.y_data = torch.tensor(target)
        self.len = self.x_data.shape[0]

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.len

train_dataset = TensorDataset(feature_data, target_data)
train_data_loader = DataLoader(train_dataset, batch_size=len(train_dataset), num_workers=16)

In [8]:
# train model
early_stop_callback = EarlyStopping(monitor='train_loss', mode='min', verbose=True, min_delta=0.001, patience=100)
trainer = pl.Trainer(callbacks=[early_stop_callback], accelerator='cuda', enable_progress_bar=True, max_epochs=10000)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [9]:
trainer.fit(model=model, train_dataloaders=train_data_loader)

Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name    | Type       | Params
---------------------------------------
0 | layer_1 | Linear     | 100   
1 | layer_2 | Linear     | 420   
2 | layer_3 | Linear     | 126   
3 | active  | ReLU       | 0     
4 | model   | Sequential | 646   
5 | loss    | MSELoss    | 0     
---------------------------------------
646       Trainable params
0         Non-trainable params
646       Total params
0.003     Total estimated model params size (MB)
/mnt/workspace/estimate_crawler_crane_ground_pressure/venv/lib/

Epoch 0: 100%|██████████| 1/1 [00:02<00:00,  2.00s/it, v_num=85]

[rank: 0] Metric train_loss improved. New best score: 22.328
[rank: 1] Metric train_loss improved. New best score: 20.100


Epoch 1: 100%|██████████| 1/1 [00:00<00:00,  1.33it/s, v_num=85]

[rank: 0] Metric train_loss improved by 11.097 >= min_delta = 0.001. New best score: 11.231
[rank: 1] Metric train_loss improved by 9.729 >= min_delta = 0.001. New best score: 10.371


...

Epoch 610: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s, v_num=85]

[rank: 0] Metric train_loss improved by 0.005 >= min_delta = 0.001. New best score: 0.016


Epoch 622: 100%|██████████| 1/1 [00:00<00:00,  1.39it/s, v_num=85]

[rank: 1] Monitored metric train_loss did not improve in the last 100 records. Best score: 0.018. Signaling Trainer to stop.


Epoch 622: 100%|██████████| 1/1 [00:00<00:00,  1.37it/s, v_num=85]


In [11]:
torch.save(model.state_dict(), './models/est_ground_pressure.pt')

In [10]:
model.eval()
for data in train_dataset:
    print(r2_score(data[1].detach().numpy() ,model(data[0]).detach().numpy()))
    print(model(data[0]).detach().numpy())
    print(data[1].detach().numpy())

0.9871332765655791
[ 3.5765715  -0.01149936  3.5738328   4.2014933  -0.02112616  3.5647104 ]
[3.97 0.   3.72 3.97 0.   3.72]
0.9977751262086295
[ 2.4848082e+00 -1.6549062e-03  3.9801366e+00  4.6294141e+00
 -4.1214563e-03  3.9723408e+00]
[2.61 0.   4.1  4.58 0.   4.1 ]
0.997350976381099
[ 8.8815498e-01 -8.0041401e-04  5.5741425e+00  4.2900510e+00
 -4.9642250e-03  5.5770316e+00]
[1.15 0.   5.59 4.13 0.   5.59]
0.994179251019459
[ 2.6416721  -0.00593635  5.1493454   2.722249   -0.01332249  5.1434326 ]
[2.93 0.   5.04 2.93 0.   5.04]
0.9984016971704133
[2.0802212  0.01571838 5.3427076  3.5589108  0.02363087 5.341245  ]
[2.16 0.   5.34 3.36 0.   5.34]
0.9973668266627363
[0.9861548  0.05872941 6.400102   3.2033217  0.09728083 6.3982706 ]
[1.25 0.   6.53 3.27 0.   6.53]
0.996679899489965
[2.4627676  0.00794786 6.505856   2.0329432  0.00753776 6.512319  ]
[2.21 0.   6.68 2.21 0.   6.68]
0.9994015860874478
[1.675635   0.03309165 6.705278   2.4884076  0.05138322 6.710149  ]
[1.83 0.04 6.75 2.46 

In [12]:
torch.onnx.export(model, torch.zeros(4), './models/est_ground_pressure.onnx')

============= Diagnostic Run torch.onnx.export version 2.0.0+cu117 =============
verbose: False, log level: Level.ERROR
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

